# Evaluation Node: Interpretability
Evaluate `evaluation_interpretability` outputs against benchmark labels and rerun this node on selected papers.

In [1]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'nodes').exists():
        break
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
if not (ROOT / 'nodes').exists():
    raise RuntimeError(f"Could not resolve project root from cwd: {Path.cwd()}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.dsrp_state import DSRPState

PILOT = ROOT / 'Pilot_Evaluation'
BENCHMARK_FILE = PILOT / 'DATA_sample_10' / 'Data Science Research Process (DSRP) Framework.csv'
RESULTS_FOLDER = PILOT / 'pilot_study_results'
VECTOR_DB_PATH = PILOT / 'pilot_study_10_vectors'
COLLECTION_NAME = 'pilot_study_10'
EMBEDDING_MODEL = 'text-embedding-3-small'
NODE_KEY = 'evaluation_interpretability'

DIMENSIONS = [
    'interpretability_discussed',
    'interpretability_approach',
    'model_transparency_level',
]

In [2]:
def normalize_text(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def normalize_paper_id(value):
    text = normalize_text(value)
    return text.lower() if text else None

def canonical_yes_no(value):
    text = normalize_text(value)
    if not text:
        return 'No'
    return 'Yes' if text.lower() in {'yes', 'true', '1', 'y'} else 'No'

def canonical_approach(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        'inherently interpretable model': 'Inherently interpretable model',
        'post-hoc explanation': 'Post-hoc explanation',
        'none reported': 'None reported',
    }
    return mapping.get(text.lower(), text)

def canonical_transparency(value):
    text = normalize_text(value)
    if not text:
        return None
    mapping = {
        'high transparency': 'High transparency',
        'moderate transparency': 'Moderate transparency',
        'medium transparency': 'Moderate transparency',
        'low transparency': 'Low transparency',
    }
    return mapping.get(text.lower(), text)

def normalize_dimension(dim, value):
    if dim == 'interpretability_discussed':
        return canonical_yes_no(value)
    if dim == 'interpretability_approach':
        return canonical_approach(value)
    if dim == 'model_transparency_level':
        return canonical_transparency(value)
    return value

def load_agent_results(results_folder):
    data = {}
    for paper_dir in sorted(results_folder.glob('*/')):
        results_file = paper_dir / 'aggregated' / 'results.json'
        if not results_file.exists():
            continue
        with open(results_file, 'r', encoding='utf-8') as f:
            parsed = json.load(f)
            pid = normalize_paper_id(parsed.get('paper_id', paper_dir.name))
            if pid:
                data[pid] = parsed.get('dsrp_outputs', {})
    return data

In [3]:
if not BENCHMARK_FILE.exists():
    raise FileNotFoundError(f"Ground truth file not found: {BENCHMARK_FILE}")

csv_mtime = pd.to_datetime(BENCHMARK_FILE.stat().st_mtime, unit='s')
print(f"Reading fresh ground truth from: {BENCHMARK_FILE}")
print(f"CSV last modified: {csv_mtime}")

benchmark_df = pd.read_csv(BENCHMARK_FILE, low_memory=False)
benchmark_df.columns = benchmark_df.columns.str.strip()
required_cols = ['Paper ID', 'Gatekeeper'] + DIMENSIONS
benchmark_eval = benchmark_df[required_cols].copy()
benchmark_eval['Paper ID'] = benchmark_eval['Paper ID'].apply(normalize_paper_id)
benchmark_eval['Gatekeeper'] = benchmark_eval['Gatekeeper'].astype(str).str.strip().str.lower()
for dim in DIMENSIONS:
    benchmark_eval[dim] = benchmark_eval[dim].apply(lambda v: normalize_dimension(dim, v))
benchmark_eval = benchmark_eval[benchmark_eval['Gatekeeper'] == 'include'].copy()

agent_data = load_agent_results(RESULTS_FOLDER)
comparison_rows = []
for _, row in benchmark_eval.iterrows():
    pid = row['Paper ID']
    if pid not in agent_data:
        continue
    node_output = agent_data[pid].get(NODE_KEY, {})
    record = {'Paper ID': pid}
    flags = []
    for dim in DIMENSIONS:
        gt = row[dim]
        if dim == 'interpretability_approach':
            raw = node_output.get('interpretability_approach', node_output.get('interpretability_method'))
        else:
            raw = node_output.get(dim)
        pred = normalize_dimension(dim, raw)
        match = gt == pred
        flags.append(match)
        record[f'GT_{dim}'] = gt
        record[f'Agent_{dim}'] = pred
        record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'
    record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
    comparison_rows.append(record)

comparison_df = pd.DataFrame(comparison_rows)
print(f'Comparable rows: {len(comparison_df)}')
show_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
print(comparison_df[show_cols].to_string(index=False))

Reading fresh ground truth from: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\Pilot_Evaluation\DATA_sample_10\Data Science Research Process (DSRP) Framework.csv
CSV last modified: 2026-04-06 15:16:36.723608017
Comparable rows: 7
 Paper ID Match_interpretability_discussed Match_interpretability_approach Match_model_transparency_level Match_All
 2018 - 8                            MATCH                           MATCH                          MATCH     MATCH
2019 - 33                            MATCH                        MISMATCH                       MISMATCH  MISMATCH
 2020 - 8                         MISMATCH                           MATCH                       MISMATCH  MISMATCH
2021 - 28                            MATCH                           MATCH                       MISMATCH  MISMATCH
2021 - 56                         MISMATCH                           MATCH                       MISMATCH  MISMATCH
2023 - 58                         MISMATCH       

In [4]:
metrics_rows = []
for dim in DIMENSIONS:
    y_true = comparison_df[f'GT_{dim}'].tolist()
    y_pred = comparison_df[f'Agent_{dim}'].tolist()
    metrics_rows.append({
        'dimension': dim,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    })
print(pd.DataFrame(metrics_rows).to_string(index=False))

                 dimension  accuracy  precision_macro  recall_macro  f1_macro
interpretability_discussed  0.428571         0.458333      0.450000  0.416667
 interpretability_approach  0.714286         0.666667      0.533333  0.583333
  model_transparency_level  0.142857         0.083333      0.166667  0.111111


## Iteration Utility

In [4]:
import importlib
import nodes.evaluation_nodes.evaluation_interpretability as node_module

def _extract_interpretability_reasoning(node_output):
    if not isinstance(node_output, dict):
        return '(no reasoning found)', ''
    reasoning = (
        node_output.get('validated_reasoning')
        or node_output.get('reasoning_explanation')
        or node_output.get('reasoning')
        or '(no reasoning found)'
    )
    audit = node_output.get('audit_commentary', '')
    return str(reasoning), str(audit or '')

def _extract_interpretability_bibliography(node_output):
    if not isinstance(node_output, dict):
        return []
    bib = node_output.get('validated_bibliography')
    if isinstance(bib, list):
        return bib
    bib = node_output.get('bibliography')
    if isinstance(bib, list):
        return bib
    return []

def _dim_key_variants(dim):
    variants = [dim]
    if dim == 'interpretability_approach':
        variants.append('interpretability_method')
    return variants

def _extract_reasoning_for_dimension(node_output, dim):
    if not isinstance(node_output, dict):
        return '(no reasoning found for this dimension)'

    # First try direct per-dimension reasoning keys.
    for key in _dim_key_variants(dim):
        candidates = [
            f'{key}_reasoning',
            f'{key}_reasoning_explanation',
            f'reasoning_{key}',
            f'explanation_{key}',
            f'{key}_explanation',
        ]
        for c in candidates:
            val = node_output.get(c)
            if isinstance(val, str) and val.strip():
                return val

    # Then try grouped containers if present.
    grouped_reasoning = node_output.get('dimension_reasoning')
    if isinstance(grouped_reasoning, dict):
        for key in _dim_key_variants(dim):
            val = grouped_reasoning.get(key)
            if isinstance(val, str) and val.strip():
                return val

    # Fall back to the global reasoning text.
    global_reasoning, _ = _extract_interpretability_reasoning(node_output)
    return global_reasoning

def _extract_bibliography_for_dimension(node_output, dim):
    if not isinstance(node_output, dict):
        return []

    # First try direct per-dimension bibliography keys.
    for key in _dim_key_variants(dim):
        candidates = [
            f'{key}_bibliography',
            f'bibliography_{key}',
        ]
        for c in candidates:
            val = node_output.get(c)
            if isinstance(val, list):
                return val

    # Then try grouped containers if present.
    grouped_bib = node_output.get('dimension_bibliography')
    if isinstance(grouped_bib, dict):
        for key in _dim_key_variants(dim):
            val = grouped_bib.get(key)
            if isinstance(val, list):
                return val

    # Fall back to global bibliography.
    return _extract_interpretability_bibliography(node_output)

def _print_bibliography_entries(bibliography):
    if not bibliography:
        print('  (none)')
        return
    for i, item in enumerate(bibliography, start=1):
        if isinstance(item, dict):
            page = item.get('page', '')
            section = item.get('section', '')
            quote = item.get('direct_quote', '')
            print(f'  [{i}] page={page} | section={section}')
            print(f'      quote: {quote}')
        else:
            print(f'  [{i}] {item}')

def run_interpretability_iteration(paper_ids, llm_model='gpt-4o-mini', verbose=True):
    importlib.reload(node_module)
    interpretability_node = node_module.evaluation_interpretability_node

    benchmark_lookup = benchmark_eval.set_index('Paper ID')
    rows = []
    raw_outputs = {}
    original_dir = Path.cwd().resolve()
    prev_env_model = os.getenv('DSRP_LLM_MODEL')

    os.chdir(ROOT)
    os.environ['DSRP_LLM_MODEL'] = llm_model

    try:
        for paper_id in paper_ids:
            pid = normalize_paper_id(paper_id)
            if not pid:
                continue

            if pid not in benchmark_lookup.index:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': 'Paper ID not found in benchmark include set'})
                continue

            state = DSRPState(
                paper_id=pid,
                dsrp_outputs={},
                collection_name=COLLECTION_NAME,
                persist_directory=str(VECTOR_DB_PATH),
                embedding_model=EMBEDDING_MODEL,
                gatekeeper={},
            )

            try:
                out = interpretability_node(state)
                dsrp_outputs = out.get('dsrp_outputs', state['dsrp_outputs'])
                node_output = dsrp_outputs.get(NODE_KEY, {})
                raw_outputs[pid] = node_output

                record = {'Paper ID': pid, 'Model': llm_model}
                flags = []
                for dim in DIMENSIONS:
                    gt = benchmark_lookup.loc[pid, dim]
                    if dim == 'interpretability_approach':
                        raw = node_output.get('interpretability_approach', node_output.get('interpretability_method'))
                    else:
                        raw = node_output.get(dim)
                    pred = normalize_dimension(dim, raw)
                    match = gt == pred
                    flags.append(match)
                    record[f'GT_{dim}'] = gt
                    record[f'New_Agent_{dim}'] = pred
                    record[f'Match_{dim}'] = 'MATCH' if match else 'MISMATCH'

                record['Match_All'] = 'MATCH' if all(flags) else 'MISMATCH'
                rows.append(record)

                if verbose:
                    summary = ' | '.join([f"{d}: {record[f'Match_{d}']}" for d in DIMENSIONS])
                    print(f"{pid}: {summary} | Match_All: {record['Match_All']}")

            except Exception as e:
                rows.append({'Paper ID': pid, 'Model': llm_model, 'Error': str(e)})
                if verbose:
                    print(f'{pid}: ERROR -> {e}')
    finally:
        if prev_env_model is None:
            os.environ.pop('DSRP_LLM_MODEL', None)
        else:
            os.environ['DSRP_LLM_MODEL'] = prev_env_model
        os.chdir(original_dir)

    return pd.DataFrame(rows), raw_outputs

def show_interpretability_label_comparison(df):
    if not isinstance(df, pd.DataFrame) or len(df) == 0:
        print('No results to display.')
        return
    cols = ['Paper ID', 'Model']
    for dim in DIMENSIONS:
        cols.extend([f'GT_{dim}', f'New_Agent_{dim}', f'Match_{dim}'])
    if 'Match_All' in df.columns:
        cols.append('Match_All')
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))

def view_existing_interpretability_reasoning(retest_df, retest_outputs, paper_id, show_bibliography=True):
    """View per-label reasoning and bibliography from existing retest results only (no rerun)."""
    pid = normalize_paper_id(paper_id)
    rows = retest_df[retest_df['Paper ID'].apply(normalize_paper_id) == pid]
    if len(rows) == 0:
        raise ValueError(f'Paper ID not found in retest_df: {paper_id}')

    row = rows.iloc[-1].to_dict()
    node_output = retest_outputs.get(pid, {})

    print('\n' + '=' * 100)
    print(f"PAPER: {row.get('Paper ID', '(missing)')} | MODEL: {row.get('Model', '(missing)')}")
    print('=' * 100)

    for dim in DIMENSIONS:
        print(f"\n{dim}")
        print(f"  GT    : {row.get(f'GT_{dim}', '(missing)')}")
        print(f"  Agent : {row.get(f'New_Agent_{dim}', '(missing)')}")
        print(f"  Match : {row.get(f'Match_{dim}', '(missing)')}")

        dim_reasoning = _extract_reasoning_for_dimension(node_output, dim)
        print('  Raw reasoning:')
        for line in str(dim_reasoning).splitlines() or ['(empty)']:
            print(f'    {line}')

        if show_bibliography:
            dim_bib = _extract_bibliography_for_dimension(node_output, dim)
            print('  Bibliography:')
            _print_bibliography_entries(dim_bib)

    print(f"\nMatch_All: {row.get('Match_All', '(missing)')}")

    _, audit = _extract_interpretability_reasoning(node_output)
    if audit.strip():
        print('\nAudit Commentary:')
        print(audit)

def rerun_single_interpretability_paper(paper_id, llm_model='gpt-4o-mini', verbose=True):
    """Rerun only one paper and return new comparison + outputs for that paper."""
    df, outputs = run_interpretability_iteration([paper_id], llm_model=llm_model, verbose=verbose)
    show_interpretability_label_comparison(df)
    return df, outputs

# Backward-compatible alias
inspect_interpretability_reasoning = view_existing_interpretability_reasoning

In [5]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7


C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MISMATCH | model_transparency_level: MATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH

Detailed label comparison (GT vs Agent):
 Paper ID       Model G

### Iteration 2: Prompt updated for Interpretability Approach and Transparency Level

In [21]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2021 - 28: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: 

### Iteration 3: Bibliography/Evidence Enforced

In [25]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2021 - 28: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH


### Iteration 4: Added Guardrail to interpretability disscussed.

In [37]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH

Detailed la

### Iteration 5: Improved Interpretability Discussion Logics

In [52]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MISMATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH

Detailed label co

re-test

In [65]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH

Detailed la

re-test

In [63]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH

Detailed label

### Iteration 6: 2021-56 interpretability discussed error fixed

In [7]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2023 - 58: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2024 - 99: interpretability_discussed: MISMATCH | interpretability_approach: MISMATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH

Detailed label comparis

### Iteration 7: Final

In [7]:
# Re-iterate on all included papers and show per-label comparison
all_included_papers = benchmark_eval['Paper ID'].tolist()
print(f'Total included papers for rerun: {len(all_included_papers)}')

retest_df, retest_outputs = run_interpretability_iteration(
    all_included_papers,
    llm_model='gpt-4o-mini',
    verbose=True,
 )

print('\nDetailed label comparison (GT vs Agent):')
show_interpretability_label_comparison(retest_df)

summary_cols = ['Paper ID'] + [f'Match_{d}' for d in DIMENSIONS] + ['Match_All']
summary_cols = [c for c in summary_cols if c in retest_df.columns]
print('\nCompact match summary:')
print(retest_df[summary_cols].to_string(index=False))

Total included papers for rerun: 7
2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2019 - 33: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2020 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 28: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2023 - 58: interpretability_discussed: MISMATCH | interpretability_approach: MATCH | model_transparency_level: MISMATCH | Match_All: MISMATCH
2024 - 99: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH

Detailed label compa

## Single Re-test


In [6]:

single_retest_df, single_retest_outputs = rerun_single_interpretability_paper(
     paper_id='2021 - 56',
     llm_model='gpt-4o-mini',
     verbose=True,
 )

C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\utils\paper_retriever.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self._vectorstore = Chroma(


2021 - 56: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
 Paper ID       Model GT_interpretability_discussed New_Agent_interpretability_discussed Match_interpretability_discussed GT_interpretability_approach New_Agent_interpretability_approach Match_interpretability_approach GT_model_transparency_level New_Agent_model_transparency_level Match_model_transparency_level Match_All
2021 - 56 gpt-4o-mini                            No                                   No                            MATCH         Post-hoc explanation                Post-hoc explanation                           MATCH            Low transparency                   Low transparency                          MATCH     MATCH


In [8]:
# Helper usage examples

# A) View per-label reasoning and bibliography from existing retest results (NO rerun)
view_existing_interpretability_reasoning(
    retest_df=retest_df,
    retest_outputs=retest_outputs,
    paper_id='2024 - 99',
    show_bibliography=True,
)



PAPER: 2024 - 99 | MODEL: gpt-4o-mini

interpretability_discussed
  GT    : Yes
  Agent : No
  Match : MISMATCH
  Raw reasoning:
    The study does not explicitly discuss interpretability, transparency, or explainability. Although it provides information about the OptiPres models, it does not employ inherently interpretable methods or post-hoc explanation techniques. The claim of 'transparent insights' does not fulfill the criteria for explicit discussion of interpretability. Therefore, the classification of Low transparency is not supported, and it should be revised to reflect a lack of evidence for any interpretability method.
  Bibliography:
  (none)

interpretability_approach
  GT    : Inherently interpretable model
  Agent : None reported
  Match : MISMATCH
  Raw reasoning:
    The study does not explicitly discuss interpretability, transparency, or explainability. Although it provides information about the OptiPres models, it does not employ inherently interpretable methods or p

In [26]:
# B) Re-run on a single paper only (optional)
single_retest_df, single_retest_outputs = rerun_single_interpretability_paper(
     paper_id='2018 - 8',
     llm_model='gpt-4o-mini',
     verbose=True,
 )

2018 - 8: interpretability_discussed: MATCH | interpretability_approach: MATCH | model_transparency_level: MATCH | Match_All: MATCH
Paper ID       Model GT_interpretability_discussed New_Agent_interpretability_discussed Match_interpretability_discussed GT_interpretability_approach New_Agent_interpretability_approach Match_interpretability_approach GT_model_transparency_level New_Agent_model_transparency_level Match_model_transparency_level Match_All
2018 - 8 gpt-4o-mini                           Yes                                  Yes                            MATCH         Post-hoc explanation                Post-hoc explanation                           MATCH           High transparency                  High transparency                          MATCH     MATCH
